# Data cleaning and canonical analysis dataset

This is the human-readable cleaning record for the GENIE BPC CRC FOLFOX paper. It shows what
was checked, what was changed, what was retained, and how many patients remain after every
cohort rule. Complex clinical rules stay in tested Python helpers; their aggregate evidence is
displayed here.

## 1. Configuration and one reproducible cleaning run

The source release is read-only. The default run refreshes `analysis/out/current/`; set
`CREATE_SNAPSHOT = True` only when a dated archival copy is needed.

In [1]:
from pathlib import Path

import pandas as pd
from IPython.display import Markdown, display

from crc_folfox.pipeline import build_clean_analysis_dataset

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_ROOT = PROJECT_ROOT / "crc_data"
OUTPUT_ROOT = PROJECT_ROOT / "analysis" / "out"
CREATE_SNAPSHOT = False

cleaning = build_clean_analysis_dataset(
    DATA_ROOT,
    OUTPUT_ROOT,
    snapshot=CREATE_SNAPSHOT,
)

display(
    pd.DataFrame(
        [
            {
                "manifest_payloads_validated": cleaning.integrity.matched_payloads,
                "analytic_tables": len(cleaning.release_audit.schema),
                "final_patients": len(cleaning.analysis_dataset),
                "final_columns": cleaning.analysis_dataset.shape[1],
            }
        ]
    )
)

,manifest_payloads_validated,analytic_tables,final_patients,final_columns
0,58,9,678,27


## 2. Source tables, expected grain, and missingness before cleaning

`expected_key` defines the logical grain of each table. Missingness is reported before cohort
construction so a reader can distinguish incomplete source fields from values lost in a merge.

In [2]:
source_audit_columns = [
    "table",
    "expected_key",
    "rows",
    "columns",
    "rows_with_any_missing",
    "columns_with_missing",
    "missing_cells",
]
display(cleaning.release_audit.schema[source_audit_columns])

,table,expected_key,rows,columns,rows_with_any_missing,columns_with_missing,missing_cells
0,patient,record_id,1485,51,1329,7,4070
1,cancer_index,record_id + ca_seq,1494,142,1494,122,96215
2,cancer_non_index,record_id + ca_seq,336,106,336,91,22176
3,regimens,record_id + regimen_number + ca_seq,5417,103,5417,68,159492
4,pathology,record_id + path_proc_number + path_rep_number,7156,320,7156,297,2052159
5,imaging,record_id + scan_number,26260,42,26260,29,594420
6,medical_oncology,record_id + md_visit_number,28164,12,8303,1,8303
7,tumor_markers,record_id + tm_number + tm_type,24462,14,430,3,871
8,cancer_panel_tests,record_id + cpt_number + ca_seq,1559,29,734,1,734


## 3. Duplicate audit and resolution

Exact duplicates and repeated logical keys are not the same thing. Longitudinal tables can
legitimately contain several observations per patient. The pipeline preserves the raw release,
applies table-specific ordering and selection rules, and requires exactly one row per patient in
the final analysis dataset.

In [3]:
display(cleaning.duplicate_audit)

,table,rows,full_row_duplicates,rows_in_duplicate_complete_keys,action
0,patient,1485,0,0,Audited at declared table grain; legitimate lo...
1,cancer_index,1494,0,0,Audited at declared table grain; legitimate lo...
2,cancer_non_index,336,0,0,Audited at declared table grain; legitimate lo...
3,regimens,5417,0,0,Audited at declared table grain; legitimate lo...
4,pathology,7156,0,0,Audited at declared table grain; legitimate lo...
5,imaging,26260,0,0,Audited at declared table grain; legitimate lo...
6,medical_oncology,28164,0,0,Audited at declared table grain; legitimate lo...
7,tumor_markers,24462,0,0,Audited at declared table grain; legitimate lo...
8,cancer_panel_tests,1559,0,0,Audited at declared table grain; legitimate lo...
9,final_analysis_dataset,678,0,0,Required one row per opaque patient key


## 4. Type, category, range, and impossible-value checks

Loading fails if required fields are absent or declared numeric fields contain nonnumeric text.
The final checks below also reject missing structural fields, invalid event indicators, negative
survival times, blank categories, surrounding whitespace, or duplicate patient keys.

In [4]:
display(cleaning.quality_checks)

category_support = cleaning.datasets.categories.groupby("variable", as_index=False).agg(
    observed_levels=("level", "nunique"),
    rare_centre_levels=("rare_within_institution", "sum"),
    absent_centre_levels=("absent_from_institution", "sum"),
)
display(category_support)

,check,failures
0,duplicate patient keys,0
1,missing structural fields,0
2,negative entry times,0
3,negative OS times,0
4,negative TTNT-D times,0
5,invalid OS events,0
6,invalid TTNT-D events,0
7,blank categorical values,0
8,categorical values with surrounding whitespace,0
9,OS-eligible rows with entry at or after exit,0


,variable,observed_levels,rare_centre_levels,absent_centre_levels
0,biologic_with_folfox,2,0,0
1,ca_d_site,14,23,8
2,ca_grade,5,6,1
3,crc_type,5,4,2
4,dmets_abdomen,2,0,0
5,dmets_bone,2,0,0
6,dmets_brain,2,1,0
7,dmets_liver,2,0,0
8,dmets_pelvis,2,0,0
9,dmets_thorax,2,0,0


## 5. Patient linkage and merge checks

Every analytic table is checked against the patient table. Institution disagreements and orphan
patient links are fatal. Cross-table joins report eligible, matched, and unmatched rows rather
than silently dropping records.

In [5]:
display(cleaning.release_audit.linkage)
display(cleaning.merge_audit)

,table,rows,patient_ids_not_in_patient_table,institution_disagreements
0,patient,1485,0,0
1,cancer_index,1494,0,0
2,cancer_non_index,336,0,0
3,regimens,5417,0,0
4,pathology,7156,0,0
5,imaging,26260,0,0
6,medical_oncology,28164,0,0
7,tumor_markers,24462,0,0
8,cancer_panel_tests,1559,0,0


,check,eligible_rows,matched_rows,unmatched_rows
0,regimen cancer links,5417,5417,0
1,cancer panel index links,1559,1559,0


## 6. Cohort filters and rows affected

The analysis population contains adults with documented metastatic colorectal cancer whose
first systemic regimen on or after metastatic presentation satisfies the locked FOLFOX rule.

In [6]:
cohort_flow = cleaning.cohort.cohort_flow.copy()
cohort_flow["removed_from_previous_step"] = (
    cohort_flow["patients"].shift(1) - cohort_flow["patients"]
).astype("Int64")
display(cohort_flow)
display(cleaning.cohort.phenotype_summary)
display(cleaning.cohort.characteristics)

,step,patients,removed_from_previous_step
0,All release patients,1485,<NA>
1,Documented advanced index CRC,1282,203
2,At least one linked post-metastatic regimen,1150,132
3,First regimen contains FOLFOX backbone,679,471
4,Primary FOLFOX definition and adult,678,1
5,Complete regimen-level OS,678,0


,regimen_class,patients
0,Exact FOLFOX,439
1,Other first regimen,395
2,FOLFOX plus biologic,239
3,CAPOX/XELOX (excluded),41
4,FOLFOXIRI (excluded),35
5,FOLFOX plus non-biologic agent,1


,characteristic,value
0,Patients,678
1,De novo Stage IV,487
2,Recurrent/metachronous,191
3,FOLFOX plus biologic,239
4,Previous oxaliplatin,19
5,Physical regimens with multiple advanced assoc...,2


## 7. Final predictors and retained missing values

Blank source cells are parsed as missing values. Genuine unknowns are not filled with a global
mean, median, or mode here: doing so before the split would let held-out institutions influence
model development. The preprocessing workflow will learn imputation and category handling from each training fold.

In [7]:
predictor_missingness = cleaning.missingness_audit.loc[
    cleaning.missingness_audit["column"].isin(cleaning.datasets.features.columns)
]
display(predictor_missingness.rename(columns={"phase_2_treatment": "cleaning_treatment"}))
display(Markdown("### Closest-prior source availability"))
display(cleaning.availability)

,column,missing_n,missing_percent,cleaning_treatment
1,institution,0,0.00,Required structural field; missing values are ...
11,age_at_folfox,0,0.00,Predictor complete; no imputation performed
12,naaccr_sex_code,0,0.00,Predictor complete; no imputation performed
13,race_ethnicity,0,0.00,Predictor complete; no imputation performed
14,metastatic_presentation,0,0.00,Predictor complete; no imputation performed
15,crc_type,1,0.15,Retained as genuine unknown; imputation deferr...
16,ca_d_site,0,0.00,Predictor complete; no imputation performed
17,ca_grade,102,15.04,Retained as genuine unknown; imputation deferr...
18,dmets_abdomen,0,0.00,Predictor complete; no imputation performed
19,dmets_bone,0,0.00,Predictor complete; no imputation performed


### Closest-prior source availability

,source,window_days,patients,percent
0,pathology,30,360,53.097345
1,imaging,30,475,70.058997
2,medical_oncology,30,531,78.318584
3,cea,30,377,55.604720
4,genomic_report,30,23,3.392330
5,pathology,90,636,93.805310
6,imaging,90,599,88.348083
7,medical_oncology,90,565,83.333333
8,cea,90,405,59.734513
9,genomic_report,90,35,5.162242


## 8. Outcome construction and endpoint eligibility

OS and TTNT-D are reconstructed and compared with the supplied release outcomes. A patient is
eligible for an endpoint only if sequencing-based delayed entry occurs before that endpoint's
exit time. Eligibility is recorded in the master dataset; it is not a train/test split.

In [8]:
display(Markdown("### OS reconstruction checks"))
display(cleaning.os_endpoint.validation)
display(Markdown("### TTNT-D reconstruction checks"))
display(cleaning.ttnt_endpoint.validation)

endpoint_summary = pd.DataFrame(
    [
        {
            "endpoint": "OS",
            "cohort_patients": len(cleaning.analysis_dataset),
            "eligible_patients": int(cleaning.analysis_dataset["os_eligible"].sum()),
            "events_among_eligible": int(
                cleaning.analysis_dataset.loc[
                    cleaning.analysis_dataset["os_eligible"], "os_event"
                ].sum()
            ),
        },
        {
            "endpoint": "TTNT-D",
            "cohort_patients": len(cleaning.analysis_dataset),
            "eligible_patients": int(cleaning.analysis_dataset["ttnt_d_eligible"].sum()),
            "events_among_eligible": int(
                cleaning.analysis_dataset.loc[
                    cleaning.analysis_dataset["ttnt_d_eligible"], "ttnt_d_event"
                ].sum()
            ),
        },
    ]
)
display(endpoint_summary)

### OS reconstruction checks

,check,patients
0,Missing OS status,0
1,Missing OS time,0
2,Negative OS time,0
3,Zero-day OS,0
4,Supplied OS differs from recomputed OS,0


### TTNT-D reconstruction checks

,check,patients
0,Missing same-cancer TTNT status,0
1,Missing same-cancer TTNT time,0
2,Negative same-cancer TTNT time,0
3,Zero-day same-cancer TTNT event,2
4,Supplied TTNT differs from reconstruction,0


,endpoint,cohort_patients,eligible_patients,events_among_eligible
0,OS,678,644,361
1,TTNT-D,678,338,264


## 9. Canonical handoff

The result is one row per patient and contains both endpoints plus eligibility flags. Downstream
modeling notebooks load this CSV through the adjacent data dictionary and select the appropriate endpoint in
memory. The cell deliberately lists only aggregate metadata and file locations.

In [9]:
assert cleaning.artifacts is not None
artifact_summary = pd.DataFrame(
    [
        {
            "artifact": path.name,
            "location": str(path.parent.relative_to(PROJECT_ROOT)),
        }
        for path in [
            cleaning.artifacts.dataset,
            cleaning.artifacts.data_dictionary,
            cleaning.artifacts.cleaning_report,
            cleaning.artifacts.run_manifest,
        ]
    ]
)
display(artifact_summary)
display(
    pd.DataFrame(
        [
            {
                "rows": len(cleaning.analysis_dataset),
                "columns": cleaning.analysis_dataset.shape[1],
                "unique_patient_keys": cleaning.analysis_dataset["patient_key"].nunique(),
                "duplicate_patient_keys": int(
                    cleaning.analysis_dataset["patient_key"].duplicated().sum()
                ),
            }
        ]
    )
)

,artifact,location
0,analysis_dataset.csv,analysis/out/current
1,data_dictionary.json,analysis/out/current
2,cleaning_report.md,analysis/out/current
3,run_manifest.json,analysis/out/current


,rows,columns,unique_patient_keys,duplicate_patient_keys
0,678,27,678,0
